# 06 — Table 1 집계

**Step 5/5 (선택)**: 여러 EVAL_DIR (다양한 dataset × N_real) 의 `results/*/splits_raw.csv` 를 긁어 paper Table 1 스타일 pivot 생성.

- 각 EVAL_DIR 은 (dataset, n_real) 하나
- `splits_raw.csv` 에 이미 per-split × per-classifier × per-config 점수가 있음
- 여기서는 긁어서 stack → pivot → Table 1 재현

필요한 것:
- 01→02/03→04 를 여러 (dataset, n_real) 조합으로 돌려둔 상태
- 각 EVAL_DIR 안에 `results/*/splits_raw.csv` 존재

## 0. Setup

In [ ]:
import os, json, re
from pathlib import Path

os.chdir('/home/work/JooKyung/TabEBM')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 60)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.width', 220)
pd.set_option('display.precision', 2)

print('ready')

## 1. EVAL_DIR 스캔 + 메타데이터

`experiments/fair_eval/` 안 모든 EVAL_DIR 나열. `results/*/splits_raw.csv` 가 있는 것만 집계 대상.

In [ ]:
fair_root = Path('experiments/fair_eval')


def pick_canonical_results_csv(eval_dir):
    """EVAL_DIR 아래 results/*/splits_raw.csv 중 canonical run 의 가장 최근 csv.

    canonical 기준 (eval_setting.json 의 is_canonical=True):
      - selected_splits == n_splits (전체)
      - classifier set == paper 6종
      - CONFIG_EXCLUDES 없음
      - per-split train-only preprocessing 사용 (paper B.3)
      - classifier/SGLD 하이퍼가 paper 기본값

    eval_setting.json 없음(legacy) 또는 is_canonical=False → 제외.
    """
    res = eval_dir / 'results'
    if not res.exists():
        return None, 'no results/'
    canonical = []
    debug_found = 0
    for subdir in sorted(res.iterdir()):
        if not subdir.is_dir():
            continue
        csv_path = subdir / 'splits_raw.csv'
        setting_path = subdir / 'eval_setting.json'
        if not csv_path.exists():
            continue
        if not setting_path.exists():
            debug_found += 1
            continue
        try:
            setting = json.loads(setting_path.read_text())
        except Exception:
            continue
        if setting.get('is_canonical', False):
            canonical.append((csv_path, subdir.stat().st_mtime))
        else:
            debug_found += 1
    if not canonical:
        return None, f'no canonical ({debug_found} debug/legacy)'
    canonical.sort(key=lambda x: x[1], reverse=True)
    return canonical[0][0], f'canonical ({debug_found} debug skipped)'


rows = []
for d in sorted(fair_root.iterdir() if fair_root.exists() else []):
    if not d.is_dir() or not (d / 'config.json').exists():
        continue
    cfg = json.loads((d / 'config.json').read_text())
    csv, status = pick_canonical_results_csv(d)
    rows.append({
        'eval_dir': d.name,
        'dataset': cfg.get('dataset'),
        'n_real': cfg.get('n_real'),
        'K': cfg.get('K'),
        'n_splits': cfg.get('n_splits'),
        'method_tag': cfg.get('method_tag', 'n/a'),
        'preprocessing': cfg.get('preprocessing', '(legacy — oracle-based)'),
        'protocol': cfg.get('split_protocol', ''),
        'csv': str(csv.relative_to(fair_root)) if csv else None,
        'status': status,
    })

meta_df = pd.DataFrame(rows)
print(f'{len(meta_df)} eval_dir scanned')
print(f'  canonical csv 있음: {meta_df["csv"].notna().sum()}개')
print(f'  debug/legacy 만: {meta_df["csv"].isna().sum()}개')
meta_df


## 2. 집계 대상 필터

- `DATASETS`: 특정 데이터셋만 (None=전부)
- `NREAL_LIST`: 특정 N_real 만 (None=전부)
- `REQUIRE_PROTOCOL`: 'paper' 만 집계할지 여부 (새 protocol 만)

In [ ]:
import fnmatch

# === EVAL_DIR 선택 (4 단계 필터; 위에서 아래로 AND) ===

# 1) 이름 직접 지정 (있으면 나머지 필터 무시)
SELECTED_EVAL_DIRS = None
# SELECTED_EVAL_DIRS = ['20260419_081846_biodeg_n100_K10',
#                      '20260419_090000_biodeg_n200_K10']

# 2) glob include (이 중 하나라도 매치)
EVAL_DIR_GLOBS = None
# EVAL_DIR_GLOBS = ['*biodeg*']                 # biodeg 전부
# EVAL_DIR_GLOBS = ['*_n100_K10', '*_n200_K10'] # n100 / n200
# EVAL_DIR_GLOBS = ['20260419_*']               # 오늘 돌린 것

# 3) glob exclude (include 뒤에 빼냄)
EVAL_DIR_EXCLUDES = None
# EVAL_DIR_EXCLUDES = ['*_K5']
# EVAL_DIR_EXCLUDES = ['20260417_*', '20260418_*']

# 4) metadata 필터 (dataset / n_real / protocol)
DATASETS    = None                 # None = 전체, or ['biodeg', 'stock']
NREAL_LIST  = None                 # None = 전체, or [20, 50, 100, 200, 500]
REQUIRE_PROTOCOL_PAPER = True      # True: paper Figure 9 protocol 만 (te_fixed 신버전)

# === 필터 적용 ===
sel = meta_df.copy()
sel = sel[sel['csv'].notna()]

if SELECTED_EVAL_DIRS is not None:
    sel = sel[sel['eval_dir'].isin(SELECTED_EVAL_DIRS)]
else:
    # include
    if EVAL_DIR_GLOBS:
        mask = sel['eval_dir'].apply(
            lambda n: any(fnmatch.fnmatch(n, pat) for pat in EVAL_DIR_GLOBS))
        sel = sel[mask]
    # exclude
    if EVAL_DIR_EXCLUDES:
        mask = sel['eval_dir'].apply(
            lambda n: not any(fnmatch.fnmatch(n, pat) for pat in EVAL_DIR_EXCLUDES))
        sel = sel[mask]

if REQUIRE_PROTOCOL_PAPER:
    sel = sel[sel['protocol'].str.contains('paper', na=False)]
if DATASETS:
    sel = sel[sel['dataset'].isin(DATASETS)]
if NREAL_LIST:
    sel = sel[sel['n_real'].isin(NREAL_LIST)]

print(f'집계 대상: {len(sel)} EVAL_DIRs')
sel[['eval_dir', 'dataset', 'n_real', 'K', 'n_splits', 'csv']]

## 3. 결과 로드 + long-form stack

각 `splits_raw.csv` 를 읽어 `(dataset, n_real, split, classifier, config, score)` long-form DataFrame 으로 쌓음.

In [ ]:
long_rows = []
for _, row in sel.iterrows():
    csv_path = fair_root / row['csv']
    raw = pd.read_csv(csv_path)
    # wide → long
    id_cols = ['n_syn', 'split', 'classifier']
    value_cols = [c for c in raw.columns if c not in id_cols]
    melt = raw.melt(id_vars=id_cols, value_vars=value_cols,
                    var_name='config', value_name='bacc')
    melt['dataset'] = row['dataset']
    melt['n_real']  = row['n_real']
    long_rows.append(melt)

if long_rows:
    long_df = pd.concat(long_rows, ignore_index=True)
    print(f'long-form rows: {len(long_df):,}')
    print(f'unique configs: {long_df["config"].nunique()}')
    print(f'unique classifiers: {long_df["classifier"].unique().tolist()}')
    print(f'n_syn values: {sorted(long_df["n_syn"].unique().tolist())}')
    display(long_df.head(8))
else:
    print('집계할 데이터 없음 — 필터 완화 또는 01-04 실행 필요')
    long_df = pd.DataFrame()

## 4. 집계할 config / n_syn / classifier 선택

Table 1 은 '각 generator × N_real' 이므로 column = config, row = (dataset, N_real) 구조.

- `CONFIGS`: 표시할 config (None = 모든 EVAL_DIR 공통만 자동 추출)
- `N_SYN_PICK`: n_syn sweep 중 어느 값으로 비교 (보통 500)
- `CLASSIFIERS_AGG`: mean 대상 classifier (None = 전부)

In [ ]:
# === Config 선택 (4 단계 필터; 04 의 CONFIG_GLOBS / EXCLUDES 와 동일 방식) ===

# 1) 이름 직접 지정 (있으면 glob 무시; baseline 항상 포함)
SELECTED_CONFIGS_OVERRIDE = None
# SELECTED_CONFIGS_OVERRIDE = ['baseline', 'tabebm_single', 'vp_b1e+06']

# 2) glob include
CONFIG_GLOBS = None
# CONFIG_GLOBS = ['tabebm_*']                     # TabEBM 계열 전부
# CONFIG_GLOBS = ['vp_b*_abF']                    # auto_beta=F 전부
# CONFIG_GLOBS = ['tabebm_single', 'vp_b1e*']     # single + large beta VP

# 3) glob exclude
CONFIG_EXCLUDES = None
# CONFIG_EXCLUDES = ['*_ivT*']                    # ignore_variance=True 제외
# CONFIG_EXCLUDES = ['tabebm_default']

# 4) 공통성 요구 — 모든 (dataset, n_real) EVAL_DIR 에 존재하는 것만
REQUIRE_COMMON = True              # False 면 합집합 (빈 칸 NaN 생김)

# === N_SYN / Classifier 선택 ===
N_SYN_PICK = None                  # None → long_df 의 최댓값 자동
# N_SYN_PICK = 500

CLASSIFIERS_AGG = None             # None = 전부
# CLASSIFIERS_AGG = ['knn', 'lr', 'rf', 'xgboost', 'mlp', 'tabpfn']  # 논문 6개
# CLASSIFIERS_AGG = ['knn', 'lr', 'rf', 'xgboost', 'mlp']            # TabPFN 제외


# === 필터 적용 ===
def common_configs(df):
    groups = df.groupby(['dataset', 'n_real'])['config'].agg(set)
    if len(groups) == 0: return set()
    return set.intersection(*groups.values)

def union_configs(df):
    return set(df['config'].unique())

if len(long_df):
    all_cfgs_pool = common_configs(long_df) if REQUIRE_COMMON else union_configs(long_df)

    if SELECTED_CONFIGS_OVERRIDE is not None:
        CONFIGS = list(SELECTED_CONFIGS_OVERRIDE)
    else:
        # include
        if CONFIG_GLOBS is None:
            cands = set(all_cfgs_pool)
        else:
            cands = {s for s in all_cfgs_pool
                     for pat in CONFIG_GLOBS if fnmatch.fnmatch(s, pat)}
        # exclude
        if CONFIG_EXCLUDES:
            cands = {s for s in cands
                     if not any(fnmatch.fnmatch(s, pat) for pat in CONFIG_EXCLUDES)}
        CONFIGS = sorted(cands)

    # baseline 항상 포함
    if 'baseline' not in CONFIGS and 'baseline' in long_df['config'].unique():
        CONFIGS = ['baseline'] + CONFIGS

    if N_SYN_PICK is None:
        N_SYN_PICK = long_df['n_syn'].max()
    if CLASSIFIERS_AGG is None:
        CLASSIFIERS_AGG = sorted(long_df['classifier'].unique())

    print(f'CONFIGS ({len(CONFIGS)}): {CONFIGS[:15]}{"..." if len(CONFIGS)>15 else ""}')
    print(f'N_SYN_PICK: {N_SYN_PICK}')
    print(f'CLASSIFIERS_AGG: {CLASSIFIERS_AGG}')
    print(f'REQUIRE_COMMON: {REQUIRE_COMMON} (pool 크기 {len(all_cfgs_pool)})')

## 5. Table 1 스타일 pivot

row = (dataset, N_real), column = config. 값은 **mean ± std across splits × classifiers** (논문 Table 1 방식).

평균 계산:
1. 각 (dataset, N_real, split, classifier, config) 에서 bacc
2. classifier 평균: 6개 classifier 값을 평균 (한 split 에 대해)
3. split 평균: 10 split 의 classifier-평균을 다시 평균 → Table 1 의 single cell
4. std 는 10 split (classifier 평균된 값) 위에서

In [ ]:
if len(long_df):
    sub = long_df[
        (long_df['config'].isin(CONFIGS)) &
        (long_df['n_syn'] == N_SYN_PICK) &
        (long_df['classifier'].isin(CLASSIFIERS_AGG))
    ].copy()

    # classifier 평균 (per split)
    per_split = sub.groupby(['dataset', 'n_real', 'split', 'config'])['bacc'].mean().reset_index()

    # split 에 대해 mean ± std
    table_mean = per_split.groupby(['dataset', 'n_real', 'config'])['bacc'].mean().unstack('config')
    table_std  = per_split.groupby(['dataset', 'n_real', 'config'])['bacc'].std().unstack('config')

    # column 순서: baseline 먼저, 나머지는 이름 순
    col_order = ['baseline'] + sorted([c for c in table_mean.columns if c != 'baseline'])
    table_mean = table_mean[col_order]
    table_std  = table_std[col_order]

    # mean ± std 합쳐서 보기 좋게
    def fmt(m, s): return f'{m:.2f}±{s:.2f}'
    table_pretty = pd.DataFrame({
        c: [fmt(table_mean.loc[idx, c], table_std.loc[idx, c])
            for idx in table_mean.index]
        for c in table_mean.columns
    }, index=table_mean.index)

    print(f'Table 1 (N_syn={N_SYN_PICK}, classifiers={CLASSIFIERS_AGG}):')
    display(table_pretty)

    # CSV 저장
    out = fair_root / f'table1_nsyn{N_SYN_PICK}.csv'
    table_pretty.to_csv(out)
    table_mean.to_csv(fair_root / f'table1_mean_nsyn{N_SYN_PICK}.csv')
    table_std.to_csv(fair_root / f'table1_std_nsyn{N_SYN_PICK}.csv')
    print(f'\nsaved: {out}')

## 6. Δ vs baseline heatmap

각 (dataset, N_real) × config 칸에 (config - baseline) pp. 녹색 = 좋음, 빨강 = 나쁨.

In [ ]:
if len(long_df) and 'baseline' in table_mean.columns:
    delta = table_mean.sub(table_mean['baseline'], axis=0).drop(columns='baseline')

    fig, ax = plt.subplots(figsize=(max(6, 0.7*len(delta.columns)+2),
                                     max(3, 0.4*len(delta)+1)))
    vmax = max(abs(delta.values).max(), 1.0)
    im = ax.imshow(delta.values, aspect='auto', cmap='RdYlGn', vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(len(delta.columns)))
    ax.set_xticklabels(delta.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(delta)))
    ax.set_yticklabels([f'{d}, n={n}' for d, n in delta.index], fontsize=8)
    for i in range(len(delta)):
        for j in range(len(delta.columns)):
            v = delta.values[i, j]
            ax.text(j, i, f'{v:+.1f}', ha='center', va='center', fontsize=7,
                    color='black' if abs(v) < vmax*0.6 else 'white')
    fig.colorbar(im, ax=ax, label='Δ vs baseline (pp)')
    ax.set_title(f'Δ vs baseline  (N_syn={N_SYN_PICK})')
    plt.tight_layout()
    fig.savefig(fair_root / f'table1_delta_heatmap_nsyn{N_SYN_PICK}.png',
                 dpi=120, bbox_inches='tight')
    plt.show()
    print(f'saved: table1_delta_heatmap_nsyn{N_SYN_PICK}.png')

## 7. Classifier 별 분해

"classifier 평균" 이전 단계 — 각 classifier 별로 (dataset, N_real, config) pivot.

In [ ]:
if len(long_df):
    per_clf = sub.groupby(
        ['dataset', 'n_real', 'classifier', 'config']
    )['bacc'].mean().unstack('config')[col_order]

    print('Classifier 별 mean balanced accuracy:')
    display(per_clf)

## 8. 논문 Table 1 baseline 수치와 비교 (biodeg 만)

논문 reported (6 classifier 평균) vs 우리 결과. 차이가 크면 protocol 재확인.

In [ ]:
# 논문 Table 1 (biodeg 행, baseline + TabEBM)
PAPER_BIODEG = pd.DataFrame({
    'n_real':     [20,    50,    100,   200,   500],
    'paper_base': [66.20, 72.66, 76.69, 80.01, 82.63],
    'paper_teb':  [69.79, 73.78, 76.45, 80.11, 82.29],
}).set_index('n_real')

if len(long_df):
    ours = table_mean.loc[table_mean.index.get_level_values('dataset') == 'biodeg']
    if len(ours):
        cmp = PAPER_BIODEG.copy()
        cmp['ours_base'] = [ours.loc[('biodeg', n), 'baseline'] if ('biodeg', n) in ours.index else np.nan
                            for n in cmp.index]
        if 'tabebm_single' in ours.columns:
            cmp['ours_teb'] = [ours.loc[('biodeg', n), 'tabebm_single']
                                if ('biodeg', n) in ours.index else np.nan for n in cmp.index]
        cmp['Δ_base'] = cmp['ours_base'] - cmp['paper_base']
        if 'ours_teb' in cmp.columns:
            cmp['Δ_teb']  = cmp['ours_teb']  - cmp['paper_teb']
        display(cmp.round(2))
    else:
        print('biodeg 결과 아직 없음')

## 9. 저장 파일 요약

In [ ]:
for p in sorted(fair_root.glob('table1*')):
    print(f'  {p.name:<40} {p.stat().st_size:>10,} bytes')